In [0]:
spark.read.option("header", True).csv("abfss://bronze@moviehistorybymg.dfs.core.windows.net/2024-12-23/movie.csv").createOrReplaceTempView("v_movie_2")

In [0]:
%sql
SELECT COUNT(1) FROM v_movie_2;

In [0]:
spark.read.option("header", True).csv("abfss://bronze@moviehistorybymg.dfs.core.windows.net/2024-12-30/movie.csv").createOrReplaceTempView("v_movie_3")

In [0]:
%sql
SELECT COUNT(1) FROM v_movie_3;

###Ingestion del archivo "movie.csv"

####Paso 1 - Leer el archivo CSV usando "DataFrameReader de Spark"

In [0]:
dbutils.widgets.help()

In [0]:
#dbutils.widgets.text("p_environment", "")
#v_environment = dbutils.widgets.get("p_environment")
dbutils.widgets.combobox("p_environment", "Producción", ["DEV", "QA", "Producción"])
v_environment = dbutils.widgets.get("p_environment")
v_environment

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configurations"

In [0]:
%run "../includes/common_functions"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

In [0]:
movie_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("releaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
])

In [0]:
movie_df = spark.read \
    .option("header", "True") \
    .schema(movie_schema) \
    .csv(f"{bronze_folder_path}/{v_file_date}/movie.csv")

In [0]:
movie_df.printSchema()

####Paso 2 - Seleccionar sólo las columnas requeridas

In [0]:
from pyspark.sql.functions import col

In [0]:
movies_selected_df = movie_df.select(col("movieId"), col("title"), col("budget"), col("popularity"), col("yearReleaseDate"), col("releaseDate"), col("revenue"), col("durationTime"), col("voteAverage"),col("voteCount"))

####Paso 3 - Cambiar el nombre de las columnas según lo requerido

In [0]:
movies_renamed_df = movies_selected_df \
                    .withColumnRenamed("movieId", "movie_id") \
                    .withColumnRenamed("yearReleaseDate", "year_release_date") \
                    .withColumnRenamed("releaseDate", "release_date") \
                    .withColumnRenamed("durationTime", "duration_time") \
                    .withColumnRenamed("voteAverage", "vote_average") \
                    .withColumnRenamed("voteCount", "vote_count") 

####Paso 4 - Agregar la columna "ingestion_date" al dataframe

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movies_final_df = add_ingestion_date(movies_renamed_df) \
                    .withColumn("enviroment", lit(v_environment)) \
                    .withColumn("file_date", lit(v_file_date))

In [0]:
display(movies_final_df)

####Paso 5 - Escribir datos en el datalake en formato "Parquet"

In [0]:
#overwrite_partition("movie_silver", "movies", "file_date", v_file_date)

In [0]:
#movies_final_df.write.mode("overwrite").partitionBy("year_release_date").parquet(f"{silver_folder_path}/movies")
#movies_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/movies")
#movies_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies")

merge_condition = 'tgt.movie_Id = src.movie_Id AND tgt.file_date = src.file_date'
merge_delta_lake (movies_final_df, "movie_silver", "movies", merge_condition, "file_date")

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movies
GROUP BY file_date;

In [0]:
%sql
DESC EXTENDED movie_silver.movies;

In [0]:
%sql
SELECT *
FROM movie_silver.movies

In [0]:
#display(spark.read.parquet(f"{silver_folder_path}/movies"))

In [0]:
#%fs
#ls abfss://silver@moviehistorybymg.dfs.core.windows.net/movies

In [0]:
#df = spark.read.parquet(f"{silver_folder_path}/movies")

In [0]:
#display(df)

In [0]:
%sql DESC EXTENDED movie_silver.movies;

In [0]:
dbutils.notebook.exit("Success")